In [1]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

from trot import driver
from trot.core.system import System
from trot.ham.chol import HamChol
from trot.ham.hubbard import HamHubbard
from trot.meas.uhf import make_uhf_meas_ops
from trot.prop import blocks, cpmc_slow
from trot.prop.types import QmcParams
from trot.trial.uhf import UhfTrial, make_uhf_trial_ops

L, n_up, n_dn, U = 100, 50, 50, 8.0
E_REF = -32.545775  # DMRG, chi = 200

# open chain
h1 = -np.eye(L, k=1) - np.eye(L, k=-1)

# UHF from a Neel guess
stag = (-1.0) ** np.arange(L)
n_a, n_b = n_up / L + 0.3 * stag, n_dn / L - 0.3 * stag
for _ in range(500):
    Ca = np.linalg.eigh(h1 + U * np.diag(n_b))[1][:, :n_up]
    Cb = np.linalg.eigh(h1 + U * np.diag(n_a))[1][:, :n_dn]
    n_a, n_b = 0.5 * (n_a + (Ca**2).sum(1)), 0.5 * (n_b + (Cb**2).sum(1))
e_uhf = np.sum(h1 * (Ca @ Ca.T + Cb @ Cb.T)) + U * (Ca**2).sum(1) @ (Cb**2).sum(1)
print(f"UHF energy: {e_uhf:.6f}")

UHF energy: -24.367344


In [2]:
ham = HamHubbard(h1=jnp.asarray(h1), u=U)
# the UHF energy kernel takes a Cholesky Hamiltonian: U n_up n_dn is one vector sqrt(U)|i><i| per site
chol = np.zeros((L, L, L))
chol[np.arange(L), np.arange(L), np.arange(L)] = np.sqrt(U)
ham_meas = HamChol(h0=jnp.zeros(()), h1=jnp.asarray(h1), chol=jnp.asarray(chol))

system = System(norb=L, nelec=(n_up, n_dn), walker_kind="unrestricted")
trial = UhfTrial(mo_coeff_a=jnp.asarray(Ca), mo_coeff_b=jnp.asarray(Cb))
params = QmcParams(n_walkers=400, dt=0.005, n_prop_steps=50, n_eql_blocks=200, n_blocks=1000,
                   weight_floor=1e-8, seed=1234)
prop_ops = cpmc_slow.make_prop_ops(ham, system.walker_kind)

result = driver.run_qmc(
    sys=system,
    params=params,
    ham_data=ham_meas,
    trial_data=trial,
    trial_ops=make_uhf_trial_ops(system),
    meas_ops=make_uhf_meas_ops(system),
    prop_ops=prop_ops,
    prop_ctx=prop_ops.build_prop_ctx(ham, None, params),
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/200]  -24.3673436658  4.000000e+02           0       0.0
[eql   40/200]  -28.6425710907  4.239046e+02           0    4197.0
[eql   80/200]  -29.0584997273  4.015806e+02           0    8334.2
[eql  120/200]  -28.7296439465  3.989239e+02           0   12586.2


KeyboardInterrupt: 

In [ ]:
# block energies: tau = 0, the equilibration blocks, then the sampling blocks (minus the outliers run_qmc rejects)
tau = np.arange(len(result.block_energies)) * params.n_prop_steps * params.dt
mean, err = float(result.mean_energy), float(result.stderr_energy)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tau, result.block_energies, lw=1, label="CPMC block energy")
ax.axhline(mean, color="C1", ls="--", label=f"CPMC: {mean:.5f} ± {err:.5f}")
ax.axhline(E_REF, color="C2", ls="--", label=f"reference: {E_REF}")
ax.axhline(e_uhf, color="grey", ls=":", label=f"UHF trial: {e_uhf:.5f}")
ax.axvline(params.n_eql_blocks * params.n_prop_steps * params.dt, color="lightgrey", zorder=0)
ax.set_xlabel("τ")
ax.set_ylabel("energy")
ax.set_title(f"UHF-trial CPMC, L = {L}, U = {U:g}", fontsize=14)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.3)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()